# Update Barton Springs WEL with interactive grid selection

This notebook loads the Barton Springs WEL file and the EBFZ grid, shows the grid on a map,
and lets you select polygons to set a new pumping rate. The updated WEL is saved locally.


In [ ]:
import shutil
import zipfile
from pathlib import Path
import urllib.request

import flopy
import geopandas as gpd
import numpy as np
from shapely.geometry import shape, LineString
from ipyleaflet import Map, GeoJSON, DrawControl, LayersControl, WidgetControl
import ipywidgets as widgets

# --- downloads ---
wel_url = (
    "https://ckan.tacc.utexas.edu/dataset/18400624-423c-42b5-ad56-6c73322584bd/"
    "resource/9c7b25c4-8cea-4965-a07a-d9b3867f18a9/"
    "download/barton_springs_2001_2010average.wel"
)
grid_url = (
    "https://ckan.tacc.utexas.edu/dataset/18400624-423c-42b5-ad56-6c73322584bd/"
    "resource/f07a257c-1d88-4819-bd5d-a104c5e3fe5b/"
    "download/ebfz_b_grid.zip"
)

wel_path = Path("barton_springs_2001_2010average.wel")
grid_zip = Path("ebfz_b_grid.zip")
grid_dir = Path("ebfz_b_grid")
grid_gdb = grid_dir / "ebfz_b_grid.gdb"
grid_layer = "ebfz_b_grid_poly101223"

def _flatten_single_dir(root: Path) -> None:
    children = [p for p in root.iterdir() if p.is_dir()]
    if len(children) == 1:
        inner = children[0]
        for item in inner.iterdir():
            shutil.move(str(item), root)
        inner.rmdir()

if not wel_path.exists():
    urllib.request.urlretrieve(wel_url, wel_path)

if not grid_gdb.exists():
    urllib.request.urlretrieve(grid_url, grid_zip)
    grid_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(grid_zip, "r") as zf:
        zf.extractall(grid_dir)
    _flatten_single_dir(grid_dir)

# --- load grid ---
gdf = gpd.read_file(grid_gdb, layer=grid_layer)
gdf = gdf.to_crs("EPSG:4326")

# --- scan WEL for grid dimensions ---
def scan_wel_metadata(path: Path):
    def strip_comment(line):
        for token in ("#", ";"):
            if token in line:
                line = line.split(token, 1)[0]
        return line.strip()

    lines = [strip_comment(line) for line in path.read_text().splitlines()]
    data_lines = [line for line in lines if line]
    data_lines.pop(0)
    nper = 0
    max_k = max_i = max_j = 1
    idx = 0
    while idx < len(data_lines):
        tokens = data_lines[idx].split()
        idx += 1
        if not tokens:
            continue
        nper += 1
        itmp = int(tokens[0])
        if itmp <= 0:
            continue
        for _ in range(itmp):
            parts = data_lines[idx].split()
            idx += 1
            k, i, j = (int(parts[0]), int(parts[1]), int(parts[2]))
            max_k = max(max_k, k)
            max_i = max(max_i, i)
            max_j = max(max_j, j)
    return nper, max_k, max_i, max_j

nper, nlay, nrow, ncol = scan_wel_metadata(wel_path)

m = flopy.modflow.Modflow(modelname="wel_read", model_ws=".")
flopy.modflow.ModflowDis(
    m,
    nlay=nlay,
    nrow=nrow,
    ncol=ncol,
    nper=nper,
    delr=1.0,
    delc=1.0,
    top=1.0,
    botm=[0.0] * nlay,
)
wel = flopy.modflow.ModflowWel.load(str(wel_path), m)


In [ ]:
# Map + selection tools
center = [float(gdf.geometry.centroid.y.median()), float(gdf.geometry.centroid.x.median())]
m = Map(center=center, zoom=9)

geojson = GeoJSON(data=gdf.__geo_interface__, name="Grid")
m.add_layer(geojson)
m.add_control(LayersControl())

selected_ids = set()
status = widgets.Label(value="Selected cells: 0")
m.add_control(WidgetControl(widget=status, position="topright"))

def _toggle_feature(feature):
    props = feature.get("properties", {})
    cell_id = int(props.get("CELL_ID"))
    if cell_id in selected_ids:
        selected_ids.remove(cell_id)
    else:
        selected_ids.add(cell_id)
    status.value = f"Selected cells: {len(selected_ids)}"

def _on_click(event, feature, **kwargs):
    _toggle_feature(feature)

geojson.on_click(_on_click)

draw = DrawControl()
draw.polygon = {"shapeOptions": {"color": "#1f77b4"}}
draw.rectangle = {"shapeOptions": {"color": "#2ca02c"}}
draw.polyline = {"shapeOptions": {"color": "#d62728"}}
draw.circle = {}

def _select_by_geometry(geom):
    hits = gdf[gdf.geometry.intersects(geom)]
    for cid in hits["CELL_ID"]:
        selected_ids.add(int(cid))
    status.value = f"Selected cells: {len(selected_ids)}"

def _on_draw(target, action, geo_json):
    geom = shape(geo_json["geometry"])
    if isinstance(geom, LineString):
        geom = geom.buffer(0.0005)  # buffer for line selection
    _select_by_geometry(geom)

draw.on_draw(_on_draw)
m.add_control(draw)
m


In [ ]:
# Apply a new pumping rate to selected cells and write updated WEL

new_rate = -20.0  # change this value
add_missing = False  # set True to add wells for selected cells not in WEL
layer_for_new = 1

spd = wel.stress_period_data.data
cell_lookup = dict(zip(gdf["CELL_ID"], zip(gdf["ROW"], gdf["COL"])))
selected_cells = {cell_lookup[cid] for cid in selected_ids if cid in cell_lookup}

new_spd = {}
for per, recs in spd.items():
    recs = recs.copy()
    mask = []
    for rec in recs:
        i = int(rec["i"])
        j = int(rec["j"])
        mask.append((i, j) in selected_cells)
    mask = np.array(mask, dtype=bool)
    recs["flux"][mask] = float(new_rate)

    if add_missing and selected_cells:
        existing = set((int(r["i"]), int(r["j"])) for r in recs)
        to_add = [cell for cell in selected_cells if cell not in existing]
        if to_add:
            new_recs = np.zeros(len(recs) + len(to_add), dtype=recs.dtype)
            new_recs[: len(recs)] = recs
            for idx, (row, col) in enumerate(to_add, start=len(recs)):
                new_recs[idx]["k"] = int(layer_for_new)
                new_recs[idx]["i"] = int(row)
                new_recs[idx]["j"] = int(col)
                new_recs[idx]["flux"] = float(new_rate)
            recs = new_recs

    new_spd[per] = recs

wel.stress_period_data = new_spd

updated_wel = Path("barton_springs_updated.wel")
wel.write_file(str(updated_wel))
print("Wrote", updated_wel)
